In [2]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER = "user_silver"
GOLD = "dim_user"

df_user = spark.table(SILVER)

df_dim = (
    df_user.select(
        "user_id",
        "name",
        "review_count",
        "average_stars",
        "fans",
        "_ingest_ts"
    )
)

if spark.catalog.tableExists(GOLD):
    delta = DeltaTable.forName(spark, GOLD)

    (
        delta.alias("t")
        .merge(
            df_dim.alias("s"),
            "t.user_id = s.user_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        df_dim.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD)
    )

StatementMeta(, 92c632a3-883d-48bd-89b7-801ad50facfa, 4, Finished, Available, Finished, False)